# 01 - Un modèle dérivé avec des VIEW

**Objectif**
- Écrire un modèle INTERLIS dérivé qui importe un modèle officiel sans le modifier, avec une VIEW `PROJECTION OF`, puis une VIEW `JOIN OF ... WHERE`.
- Compiler le modèle avec `interlis build`, puis produire le SQL avec `interlis convert-sql`.
- Lire le `CREATE VIEW` obtenu et le relier à la définition INTERLIS.

Cas d'étude : le modèle minimal « Planungszonen » (zones réservées, jeu de géodonnées de base n° 76, ARE), comme dans l'atelier du 27 mars 2025.

## Pourquoi une VIEW

Un modèle officiel est normalisé : références vers des catalogues, structures imbriquées, textes multilingues. Un géoservice attend une table plate par type de géométrie, avec les libellés déjà résolus. La VIEW (INTERLIS 2.3, section 3.15 du manuel de référence) décrit cette mise à plat dans un modèle dérivé : à partir de quelles classes, avec quels attributs, avec quelle condition de jointure.

```text
modèle officiel  ->  modèle dérivé (VIEW)  ->  vue SQL  ->  collection OGC API
```

In [ ]:
!mkdir -p modeles sorties

## Étape 1 - Le modèle de base

Le modèle officiel est `Planungszonen_V1_1` ([source](https://models.geo.admin.ch/ARE/Planungszonen_V1_1.ili)). Dans le thème `Geobasisdaten` :

- `Planungszone` : une surface (`Geometrie`), `Rechtsstatus`, `publiziertAb`, `publiziertBis`, `Bemerkungen` ;
- `Typ_Planungszone` : `Code`, `Bezeichnung`, `Abkuerzung`, `Festlegung_Stufe` ;
- l'association `TypPZ_Planungszone` : chaque zone référence un type par le rôle `TypPZ`.

Le modèle dérivé n'ajoute aucune donnée : il décrit une lecture de ces classes.

## Étape 2 - Écrire le modèle dérivé avec une VIEW PROJECTION OF

Un modèle dérivé est un `MODEL` qui importe le modèle officiel (`IMPORTS`) et déclare un `VIEW TOPIC` qui étend le thème de base. `PROJECTION OF` reprend une seule classe. Chaque attribut de la vue s'écrit `nom := Alias -> attribut` ; une flèche supplémentaire suit une référence.

La cellule suivante écrit le modèle dans `sorties/projection.ili`.

In [ ]:
%%writefile sorties/projection.ili
INTERLIS 2.3;

MODEL Planungszonen_Views (de)
  AT "mailto:atelier@example.org"
  VERSION "2025-03-27" =

  IMPORTS Planungszonen_V1_1;

  VIEW TOPIC Views EXTENDS Planungszonen_V1_1.Geobasisdaten =
    VIEW zonen_projektion
      PROJECTION OF Z ~ Planungszonen_V1_1.Geobasisdaten.Planungszone;
      =
      ATTRIBUTE
        rechtsstatus := Z -> Rechtsstatus;
        publiziertab := Z -> publiziertAb;
        geometrie := Z -> Geometrie;
    END zonen_projektion;

  END Views;

END Planungszonen_Views.

## Étape 3 - Télécharger le modèle de base, puis compiler

`fetch-models` lit les `IMPORTS` du modèle dérivé et télécharge, de proche en proche, les modèles manquants dans `-o`. Ce dossier sert ensuite d'option `--repo` aux autres commandes, qui ne vont jamais sur le réseau.

`build -q` construit le modèle en mémoire (`-q` masque les avertissements sur les cas de la grammaire non encore couverts) et en affiche l'arbre, ici filtré sur les lignes `View Name`. Un code de sortie 0 signifie que le modèle est lisible et que tous ses renvois sont résolus.

In [ ]:
!interlis fetch-models sorties/projection.ili -o modeles

In [ ]:
!interlis build -q sorties/projection.ili --repo modeles | grep "View Name"

## Étape 4 - Générer le SQL et lire le CREATE VIEW

`convert-sql` produit le schéma de base de données. Les classes du modèle importé qu'une VIEW utilise sont reprises comme tables (disposition compatible ili2db : `t_id`, `t_ili_tid`, clés étrangères par `t_id`). Chaque VIEW devient un `CREATE VIEW`.

Le code de sortie 2 signifie « terminé avec remarques » : ici, une surface qui admet des arcs est stockée en géométrie linéaire.

In [ ]:
!interlis convert-sql sorties/projection.ili --repo modeles -o sorties/projection.sql; echo "code de sortie : $?"

In [ ]:
!grep -o 'CREATE TABLE "[a-z_]*"' sorties/projection.sql
!sed -n '/CREATE VIEW "zonen_projektion"/,/;/p' sorties/projection.sql

Lecture : la vue `zonen_projektion` sélectionne trois colonnes de la table `planungszone`, sous le nom donné par la VIEW, plus les deux identifiants de l'objet.

### À vous : une VIEW sur les types de zone

Compléter `sorties/typen.ili` : une VIEW `typen` de type `PROJECTION OF` sur la classe `Planungszonen_V1_1.Geobasisdaten.Typ_Planungszone`, avec les attributs `code`, `bezeichnung` et `festlegung_stufe` (respectivement `Code`, `Bezeichnung`, `Festlegung_Stufe`). S'inspirer de la VIEW de l'étape 2. Remplacer les `...`, puis lancer la cellule de commande qui suit.

In [ ]:
%%writefile sorties/typen.ili
INTERLIS 2.3;

MODEL Planungszonen_Views (de)
  AT "mailto:atelier@example.org"
  VERSION "2025-03-27" =

  IMPORTS Planungszonen_V1_1;

  VIEW TOPIC Views EXTENDS Planungszonen_V1_1.Geobasisdaten =
    VIEW typen
      PROJECTION OF T ~ ...;
      =
      ATTRIBUTE
        ...
    END typen;

  END Views;

END Planungszonen_Views.

In [ ]:
!interlis convert-sql sorties/typen.ili --repo modeles -o sorties/typen.sql
!sed -n '/CREATE VIEW "typen"/,/;/p' sorties/typen.sql

**Vérification** : la cellule suivante dit si l'exercice est réussi.

In [ ]:
sql = open("sorties/typen.sql").read()
assert 'CREATE VIEW "typen"' in sql, "Aucun CREATE VIEW \"typen\" dans le SQL"
for colonne in ("code", "bezeichnung", "festlegung_stufe"):
    assert f'AS "{colonne}"' in sql, f"colonne manquante : {colonne}"
print("Exercice réussi.")

#### Solution

À ouvrir après avoir essayé. Exécuter la cellule remplace la réponse de l'exercice ; relancer ensuite la vérification.

In [ ]:
%%writefile sorties/typen.ili
INTERLIS 2.3;

MODEL Planungszonen_Views (de)
  AT "mailto:atelier@example.org"
  VERSION "2025-03-27" =

  IMPORTS Planungszonen_V1_1;

  VIEW TOPIC Views EXTENDS Planungszonen_V1_1.Geobasisdaten =
    VIEW typen
      PROJECTION OF T ~ Planungszonen_V1_1.Geobasisdaten.Typ_Planungszone;
      =
      ATTRIBUTE
        code := T -> Code;
        bezeichnung := T -> Bezeichnung;
        festlegung_stufe := T -> Festlegung_Stufe;
    END typen;

  END Views;

END Planungszonen_Views.

## Étape 5 - La VIEW JOIN OF avec WHERE

Un service veut souvent, pour chaque zone, le libellé de son type. `JOIN OF` réunit plusieurs classes dans une seule vue ; la clause `WHERE` dit comment les objets se correspondent :

```text
JOIN OF Z ~ ...Planungszone, T ~ ...Typ_Planungszone;
WHERE Z -> TypPZ == T;
```

`Z -> TypPZ` est la référence de la zone vers son type ; `== T` la compare à l'objet `T`. En INTERLIS, une jointure sans `WHERE` est un produit cartésien : à écrire systématiquement.

In [ ]:
%%writefile sorties/jointure.ili
INTERLIS 2.3;

MODEL Planungszonen_Views (de)
  AT "mailto:atelier@example.org"
  VERSION "2025-03-27" =

  IMPORTS Planungszonen_V1_1;

  VIEW TOPIC Views EXTENDS Planungszonen_V1_1.Geobasisdaten =
    VIEW zonen
      JOIN OF Z ~ Planungszonen_V1_1.Geobasisdaten.Planungszone,
              T ~ Planungszonen_V1_1.Geobasisdaten.Typ_Planungszone;
      WHERE Z -> TypPZ == T;
      =
      ATTRIBUTE
        typ_code := T -> Code;
        typ_bezeichnung := T -> Bezeichnung;
        rechtsstatus := Z -> Rechtsstatus;
        geometrie := Z -> Geometrie;
    END zonen;

  END Views;

END Planungszonen_Views.

In [ ]:
!interlis convert-sql sorties/jointure.ili --repo modeles -o sorties/jointure.sql
!sed -n '/CREATE VIEW "zonen"/,/;/p' sorties/jointure.sql

Lecture : la vue sélectionne dans `planungszone` et `typ_planungszone` ; la condition `WHERE` devient l'égalité entre la clé étrangère `typpz` et le `t_id` du type. L'identifiant `t_id` de la vue concatène les deux identifiants, car une ligne de la vue correspond à un couple d'objets.

### Sans WHERE

Que fait `convert-sql` si la clause est omise ? Le modèle suivant est identique, sans `WHERE`.

In [ ]:
%%writefile sorties/sans_where.ili
INTERLIS 2.3;

MODEL Planungszonen_Views (de)
  AT "mailto:atelier@example.org"
  VERSION "2025-03-27" =

  IMPORTS Planungszonen_V1_1;

  VIEW TOPIC Views EXTENDS Planungszonen_V1_1.Geobasisdaten =
    VIEW zonen
      JOIN OF Z ~ Planungszonen_V1_1.Geobasisdaten.Planungszone,
              T ~ Planungszonen_V1_1.Geobasisdaten.Typ_Planungszone;
      =
      ATTRIBUTE
        typ_code := T -> Code;
        typ_bezeichnung := T -> Bezeichnung;
        rechtsstatus := Z -> Rechtsstatus;
        geometrie := Z -> Geometrie;
    END zonen;

  END Views;

END Planungszonen_Views.

In [ ]:
!interlis convert-sql sorties/sans_where.ili --repo modeles -o sorties/sans_where.sql

`convert-sql` signale une remarque (`SQL-VIEW-JOIN-ASSOCIATION`) : il a déduit la jointure de l'association du modèle. Cette déduction n'est possible que lorsque les deux classes sont liées directement. La sortie JSON-FG (notebook 03) applique, elle, la définition INTERLIS à la lettre, c'est-à-dire un produit cartésien. Écrire un `WHERE` explicite évite l'écart.

### À vous : une jointure filtrée

Compléter `sorties/kanton.ili` : une VIEW `zonen_kanton` de type `JOIN OF` sur les mêmes classes, qui ne garde que les zones dont le type a `Festlegung_Stufe` égal à `#Kanton`. Attributs : `typ_bezeichnung` (`T -> Bezeichnung`), `rechtsstatus` (`Z -> Rechtsstatus`) et `geometrie`.

Indice : plusieurs conditions se combinent avec `AND` ; une valeur d'énumération s'écrit avec `#`.

In [ ]:
%%writefile sorties/kanton.ili
INTERLIS 2.3;

MODEL Planungszonen_Views (de)
  AT "mailto:atelier@example.org"
  VERSION "2025-03-27" =

  IMPORTS Planungszonen_V1_1;

  VIEW TOPIC Views EXTENDS Planungszonen_V1_1.Geobasisdaten =
    VIEW zonen_kanton
      JOIN OF ...;
      WHERE ...;
      =
      ATTRIBUTE
        ...
    END zonen_kanton;

  END Views;

END Planungszonen_Views.

In [ ]:
!interlis convert-sql sorties/kanton.ili --repo modeles -o sorties/kanton.sql
!sed -n '/CREATE VIEW "zonen_kanton"/,/;/p' sorties/kanton.sql

**Vérification** : la cellule suivante dit si l'exercice est réussi.

In [ ]:
vue = open("sorties/kanton.sql").read()
assert 'CREATE VIEW "zonen_kanton"' in vue, "Aucun CREATE VIEW \"zonen_kanton\" dans le SQL"
assert '"z"."typpz" = "t"."t_id"' in vue, "la jointure entre zone et type manque"
assert "'Kanton'" in vue, "le filtre sur Festlegung_Stufe manque"
print("Exercice réussi.")

#### Solution

À ouvrir après avoir essayé. Exécuter la cellule remplace la réponse de l'exercice ; relancer ensuite la vérification.

In [ ]:
%%writefile sorties/kanton.ili
INTERLIS 2.3;

MODEL Planungszonen_Views (de)
  AT "mailto:atelier@example.org"
  VERSION "2025-03-27" =

  IMPORTS Planungszonen_V1_1;

  VIEW TOPIC Views EXTENDS Planungszonen_V1_1.Geobasisdaten =
    VIEW zonen_kanton
      JOIN OF Z ~ Planungszonen_V1_1.Geobasisdaten.Planungszone,
              T ~ Planungszonen_V1_1.Geobasisdaten.Typ_Planungszone;
      WHERE Z -> TypPZ == T AND T -> Festlegung_Stufe == #Kanton;
      =
      ATTRIBUTE
        typ_bezeichnung := T -> Bezeichnung;
        rechtsstatus := Z -> Rechtsstatus;
        geometrie := Z -> Geometrie;
    END zonen_kanton;

  END Views;

END Planungszonen_Views.

## À retenir

- Le modèle officiel est importé sans modification ; la VIEW est un modèle à part.
- `PROJECTION OF` : une classe, avec renommage et résolution des références.
- `JOIN OF ... WHERE` : plusieurs classes dans une table plate ; écrire toujours le `WHERE`.
- `build` valide le modèle, `convert-sql` produit tables et `CREATE VIEW`.
- Le code de sortie 2 signale un succès avec remarques ; il n'est pas une erreur.

## Suite

Notebook `02_representation.ipynb` : une règle graphique `GRAPHIC` construite sur une VIEW, et ses signatures.